In [ ]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

PROJECT_ROOT = "/Users/jun/GitStudy/human_A"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from preprocessing import step1_prepare_window_data
from inference_core import get_alarm_status, calculate_rca

NB_DIR = PROJECT_ROOT
DATA_CSV = "/Users/jun/GitStudy/human_A/data/generated_test_data_0420.csv"
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

print(f"CSV: {DATA_CSV}")
print(f"MODELS: {MODELS_DIR}")


In [ ]:
df_raw = pd.read_csv(DATA_CSV)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"])
df_raw = df_raw.set_index("timestamp")
df_agg, _ = step1_prepare_window_data(df_raw, window_method="tumbling")
df_agg = df_agg.dropna()
print(f"1min raw: {df_raw.shape}  →  10min agg: {df_agg.shape}")


In [ ]:
config_files = glob.glob(os.path.join(MODELS_DIR, "*_config.json"))
SYSTEMS = sorted(os.path.basename(f).replace("_config.json", "") for f in config_files)

MODELS_DATA = {}
for dom in SYSTEMS:
    model_path = os.path.join(MODELS_DIR, f"{dom}_model.keras")
    scaler_path = os.path.join(MODELS_DIR, f"{dom}_scaler.pkl")
    config_path = os.path.join(MODELS_DIR, f"{dom}_config.json")
    if not all(os.path.exists(p) for p in [model_path, scaler_path, config_path]):
        continue
    MODELS_DATA[dom] = {
        "model": tf.keras.models.load_model(model_path),
        "scaler": joblib.load(scaler_path),
        "config": json.load(open(config_path, "r", encoding="utf-8")),
    }

domain_results = {}
for dom, data in MODELS_DATA.items():
    cfg = data["config"]
    features = cfg["features"]
    t_caut = cfg["threshold_caution"]
    t_warn = cfg["threshold_warning"]
    t_err = cfg.get("threshold_critical", cfg.get("threshold_error"))
    X = pd.DataFrame(index=df_agg.index, columns=features, dtype=float)
    for f in features:
        X[f] = df_agg[f].astype(float).values if f in df_agg.columns else 0.0
    X_scaled = data["scaler"].transform(X)
    preds = data["model"].predict(X_scaled, batch_size=512, verbose=0)
    errors = np.power(X_scaled - preds, 2)
    mse = errors.mean(axis=1)
    domain_results[dom] = {
        "mse": mse, "errors": errors, "features": features,
        "t_caut": float(t_caut), "t_warn": float(t_warn), "t_err": float(t_err),
    }

records = []
for i, ts in enumerate(df_agg.index):
    flat = {"timestamp": ts, "overall_alarm_level": 0}
    for dom, d in domain_results.items():
        mse_i = float(d["mse"][i])
        lvl, _ = get_alarm_status(mse_i, d["t_caut"], d["t_warn"], d["t_err"])
        flat[f"{dom}_level"] = lvl
        if lvl > flat["overall_alarm_level"]:
            flat["overall_alarm_level"] = lvl
    records.append(flat)

df_results = pd.DataFrame(records).set_index("timestamp")
domains = sorted({
    c[:-len("_level")]
    for c in df_results.columns
    if c.endswith("_level") and not c.startswith("overall_")
})
print(f"df_results: {df_results.shape}, domains: {domains}")


In [14]:
# ── 이상 감지된 구간만 필터링 → 4개 도메인 alarm_level 표 ──────────────
# 4개 도메인 중 어느 하나라도 level > 0 이면 타임스탬프 기록
events = df_results[df_results["overall_alarm_level"] > 0].copy()
print(f"🚨 이상 감지된 타임스탬프 수: {len(events):,}개")

# 도메인 이름 → 축약 헤더 매핑 (user-friendly)
ABBR = {
    "hydraulic": "hyd",
    "motor":     "mot",
    "nutrient":  "nut",
    "zone_drip": "zone",
    "salt":      "salt",   # 레거시 결과 호환
}

# 4개 도메인 level 만 추려서 깔끔한 표 구성
anomaly_table = pd.DataFrame(index=events.index)
for dom in domains:
    header = ABBR.get(dom, dom)
    anomaly_table[header] = events[f"{dom}_level"].astype(int)

anomaly_table.index.name = "timestamp"
print(f"\n📋 이상 감지 구간 (상위 30개):")
display(anomaly_table.head(30))

# CSV 로도 저장
OUT_DIR = os.path.join(NB_DIR, "anomaly_review_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
anomaly_csv = os.path.join(OUT_DIR, "anomaly_timestamps.csv")
anomaly_table.to_csv(anomaly_csv, encoding="utf-8-sig")
print(f"\n💾 저장 완료: {anomaly_csv}")

🚨 이상 감지된 타임스탬프 수: 8,790개

📋 이상 감지 구간 (상위 30개):


,hyd,mot,nut,zone
timestamp,,,,
2026-03-01 04:20:00,0,0,0,1
2026-03-01 06:00:00,3,3,0,0
2026-03-01 06:10:00,3,2,0,0
2026-03-01 06:20:00,1,0,0,0
2026-03-01 14:10:00,0,0,1,0
2026-03-01 18:00:00,3,0,0,1
2026-03-01 18:10:00,3,0,0,0
2026-03-01 18:20:00,2,0,0,0
2026-03-01 18:40:00,0,0,1,0



💾 저장 완료: c:\Users\ui203\OneDrive\문서\green\human_A2\models\anomaly_review_outputs\anomaly_timestamps.csv
